In [4]:
# ── Ячейка 1: импорты и инфраструктура ────────────────────────────────────────
import sys; sys.path.insert(0, "..")
import gc, time
import numpy as np
import joblib

from src import config, data, preprocess, evaluate

con = data.get_duckdb_connection()
feature_cols = data.get_feature_cols(con)   # assert: 76 признаков, таргет не утёк

n_train = data.count_rows(con, config.TRAIN_PATH)
n_val   = data.count_rows(con, config.VAL_PATH)
n_test  = data.count_rows(con, config.TEST_PATH)

print(f"Признаков (raw): {len(feature_cols)}")
print(f"train={n_train:,}  val={n_val:,}  test={n_test:,}")

Признаков (raw): 76
train=9,583,883  val=2,053,688  test=2,053,697


In [5]:
# ── Ячейка 2: препроцессор (linear/nn-view) ───────────────────────────────────
# Фитится на подвыборке train, сохраняется в models/preprocessor.pkl.
# Если уже сохранён (мы его строили при проверке) — просто грузим.
PRE_PATH = config.MODELS_DIR / "preprocessor.pkl"

if PRE_PATH.exists():
    pre = preprocess.load_preprocessor(PRE_PATH)
    print(f"Препроцессор загружен ← {PRE_PATH}")
else:
    pre, PRE_PATH = preprocess.fit_preprocessor(con, feature_cols, sample_size=2_000_000)
    print(f"Препроцессор обучен и сохранён → {PRE_PATH}")

print(f"Выходных признаков linear-view: {pre.n_features_out_} (ожидаем 80)")

Препроцессор загружен ← /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/models/preprocessor.pkl
Выходных признаков linear-view: 80 (ожидаем 80)


In [6]:
# ── Ячейка 3: загрузка train в RAM (linear-view) ──────────────────────────────
# LogisticRegression(saga) учится на полном train целиком (RAM позволяет).
# Применяем препроцессор побатчно при загрузке, чтобы не держать две копии.
print("Загружаем train (preprocess → linear-view)...")
t0 = time.time()

X_chunks, y_chunks = [], []
for X_b, y_b in data.iter_batches(config.TRAIN_PATH, feature_cols):
    X_chunks.append(pre.transform(X_b))   # (batch, 80) float32
    y_chunks.append(y_b)
    del X_b; gc.collect()

X_train = np.concatenate(X_chunks); del X_chunks
y_train = np.concatenate(y_chunks); del y_chunks
gc.collect()

print(f"X_train: {X_train.shape}  {X_train.nbytes/1024**3:.2f} GB")
print(f"malicious={y_train.mean():.4f}  |  загружено за {time.time()-t0:.1f}s")

Загружаем train (preprocess → linear-view)...
X_train: (9583883, 80)  2.86 GB
malicious=0.2696  |  загружено за 17.4s


In [ ]:
# ── Ячейка 4: обучение LogisticRegression(saga) ───────────────────────────────
from sklearn.linear_model import LogisticRegression

# saga: поддерживает L2 + большие данные, даёт калиброванные вероятности
# (в отличие от SGDClassifier, у которого они вырождались в 0/1).
# class_weight='balanced' — компенсация дисбаланса 73/27.
# Данные уже масштабированы препроцессором → сходимость быстрая.
model = LogisticRegression(
    solver="saga",
    C=1.0,
    class_weight="balanced",
    max_iter=200,
    random_state=config.RANDOM_STATE,
    verbose=1,
)

print("Обучение LogisticRegression(saga)...")
t0 = time.time()
model.fit(X_train, y_train)
print(f"\nГотово за {time.time()-t0:.1f}s  |  итераций: {model.n_iter_[0]}  |  "
      f"сошлось: {model.n_iter_[0] < model.max_iter}")

del X_train, y_train; gc.collect()

joblib.dump(model, config.MODELS_DIR / "logreg.pkl")
print(f"Модель сохранена → {config.MODELS_DIR / 'logreg.pkl'}")

Обучение LogisticRegression(saga)...


/home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Epoch 1, change: 1
Epoch 2, change: 0.40010566
Epoch 3, change: 0.13489743
Epoch 4, change: 0.019429222
Epoch 5, change: 0.0060007288
Epoch 6, change: 0.0056341849
Epoch 7, change: 0.0015137175
Epoch 8, change: 0.001493812
Epoch 9, change: 0.0015077206
Epoch 10, change: 0.00023671699
convergence after 11 epochs took 114 seconds

Готово за 115.2s  |  итераций: 11  |  сошлось: True
Модель сохранена → /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/models/logreg.pkl


In [8]:
# ── Ячейка 5: предикт-функция и оценка на VAL (подбор порога) ──────────────────
# predict_proba_fn инкапсулирует препроцессинг: на вход сырой батч (76),
# внутри transform → linear-view (80) → вероятность класса 1.
def predict_logreg(X_raw):
    return model.predict_proba(pre.transform(X_raw))[:, 1]

# VAL: threshold=None → подбор оптимума по PR-кривой, порог возвращается.
m_val = evaluate.evaluate(
    "logreg", predict_logreg, config.VAL_PATH, feature_cols,
    threshold=None, split_name="val",
)
best_thr = m_val["threshold"]
print(f"\nПодобранный на val порог: {best_thr}")


  logreg  (val)
строк 2,053,688 за 3.58s  |  порог 0.6175 (tuned_on_this_split)
F1_macro 0.9689  MCC 0.9392  PR-AUC 0.978792  ROC-AUC 0.993683  FPR@95 0.032117
TN=1,450,893  FP=49,107  FN=2,581  TP=551,107  (FPR=0.0327, FNR=0.0047)

per-class recall:
  Benign                           n=1,500,000  recall=96.73%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=100.00%
  DDoS LOIC-HTTP                   n=  43,399  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,832  recall=100.00%
  Bot                              n=  14,423  recall=98.84%
  SSH-Patator                      n=  13,897  recall=100.00%
  DoS GoldenEye                    n=   4,029  recall=78.53%
  DoS Slowloris                    n=   1,541  recall=4.61% ◄
  DDoS LOIC-UDP                    n=     357  recall=95.52%
  Web Attack - Brute Force         n=      39  recall=0.00% ◄
  Web A

In [9]:
# ── Ячейка 6: оценка на TEST (замороженный порог) ─────────────────────────────
# threshold=best_thr → порог НЕ переоптимизируется на test (честная оценка).
m_test = evaluate.evaluate(
    "logreg", predict_logreg, config.TEST_PATH, feature_cols,
    threshold=best_thr, split_name="test",
)


  logreg  (test)
строк 2,053,697 за 3.4s  |  порог 0.6175 (frozen_from_val)
F1_macro 0.9684  MCC 0.9383  PR-AUC 0.97197  ROC-AUC 0.993043  FPR@95 0.032697
TN=1,450,037  FP=49,963  FN=2,538  TP=551,159  (FPR=0.0333, FNR=0.0046)

per-class recall:
  Benign                           n=1,500,000  recall=96.67%
  DoS Hulk                         n= 270,445  recall=100.00%
  DDoS HOIC                        n= 161,157  recall=100.00%
  DDoS LOIC-HTTP                   n=  43,400  recall=100.00%
  FTP-Patator                      n=  28,545  recall=100.00%
  DoS Slowhttptest                 n=  15,833  recall=100.00%
  Bot                              n=  14,424  recall=98.83%
  SSH-Patator                      n=  13,898  recall=100.00%
  DoS GoldenEye                    n=   4,030  recall=76.72%
  DoS Slowloris                    n=   1,542  recall=13.16% ◄
  DDoS LOIC-UDP                    n=     358  recall=92.46%
  Web Attack - Brute Force         n=      39  recall=0.00% ◄
  Web Attac